In [60]:
from pydantic import BaseModel
from typing import List


class PreferenceExample(BaseModel):

    prompt: str

    chosen: str

    rejected: str

    failure_capabilities: List[str]

    primary_failure: str

    severity: str

In [61]:
import os 
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [62]:
import json

with open("investigation_reports.json") as f:

    investigation_reports = json.load(f)

In [63]:
investigation_reports

[{'metadata': {'task_type': None,
   'specialty': None,
   'difficulty': None,
   'dataset': None,
   'evaluator_version': 'v1'},
  'prediction_summary': {'concise_prediction': 'The correct answer is A. Disclose the error to the patient and put it in the operative report',
   'confidence': None},
  'observations': [{'source': 'Ground Truth',
    'text': 'The correct answer is B. Tell the attending that he cannot fail to disclose this mistake'},
   {'source': 'Prediction',
    'text': 'The correct answer is A. Disclose the error to the patient and put it in the operative report'}],
  'failure_tags': ['Clinical Reasoning', 'Ethics', 'Communication'],
  'inferences': [{'type': 'Reasoning',
    'reasoning': 'The model prediction suggests that the resident should disclose the error to the patient and put it in the operative report, which may not be the most appropriate next step in this situation.',
    'supporting_observations': ['The attending physician instructed the resident not to disc

In [64]:
import json
from groq import Groq
from pydantic import BaseModel

client = Groq(api_key=GROQ_API_KEY)


# ============================================================
# LLM Response
# ============================================================

class PreferenceResponse(BaseModel):

    chosen: str


# ============================================================
# Prompt
# ============================================================

PREFERENCE_GENERATOR_PROMPT = """
You are a board-certified physician.

Your task is to generate a high-quality corrected response for post-training a medical language model.

You are given:

1. Clinical Question
2. Model Prediction
3. Investigation Report

The Investigation Report identifies unsupported claims, missing information,
safety concerns, guideline violations, and the primary failure.

Your goal is to produce the BEST possible clinical response.

This is NOT an evaluation task.

This is NOT a minimal editing task.

If the prediction is substantially incorrect, rewrite it completely.

If the prediction is partially correct, preserve only the correct clinical information and replace everything else.

Never preserve unsupported or unsafe reasoning.

Your corrected response should:

• Be clinically accurate
• Follow current medical guidelines
• Include all clinically important information
• Remove unsupported claims
• Correct unsafe recommendations
• Improve clinical reasoning
• Be concise, complete, and natural

Do NOT mention:

- the investigation report
- evaluation
- unsupported claims
- safety analysis
- guidelines
- failure analysis

Preserve the response format expected by the task.

Examples

• If the task is multiple-choice, return the correct option followed by a brief clinical justification.

• If the task is free-text QA, return a complete answer.

• If the task is report generation, return the corrected report.

• If the task is summarization, return the corrected summary.

The corrected response should be suitable as the preferred response in a DPO dataset.

It should be a response that an expert physician would write.

Do not return only a single option letter unless the original prediction itself only consisted of a single letter.

Return ONLY valid JSON.

{
    "chosen": "..."
}
"""


# ============================================================
# Preference Generator
# ============================================================

class PreferenceGenerator:

    def __init__(

        self,

        model="llama-3.3-70b-versatile"

    ):

        self.model = model

    def generate(

        self,

        sample: dict

    ) -> PreferenceExample:

        question = sample["question"]

        prediction = sample["prediction"]

        investigation = sample["investigation"]


        user_prompt = f"""
QUESTION

{question}

------------------------------------------------

MODEL PREDICTION

{prediction}

------------------------------------------------

INVESTIGATION REPORT

{json.dumps(investigation, indent=2)}
"""


        response = client.chat.completions.create(

            model=self.model,

            temperature=0,

            response_format={

                "type": "json_object"

            },

            messages=[

                {

                    "role": "system",

                    "content": PREFERENCE_GENERATOR_PROMPT

                },

                {

                    "role": "user",

                    "content": user_prompt

                }

            ]

        )


        data = json.loads(

            response.choices[0].message.content

        )

        parsed = PreferenceResponse.model_validate(

            data

        )


        return PreferenceExample(

            prompt=question,

            chosen=parsed.chosen,

            rejected=prediction,

            failure_capabilities=investigation["failure_tags"],

            primary_failure=investigation["primary_failure"]["category"],

            severity=investigation["safety"]["severity"]

        )

In [65]:
from typing import List


class PreferenceDatasetBuilder:

    def __init__(self, generator: PreferenceGenerator):

        self.generator = generator

    def build(

        self,

        investigation_reports: List[dict]

    ) -> List[PreferenceExample]:

        preferences = []

        total = len(investigation_reports)

        for i, sample in enumerate(investigation_reports, start=1):

            print(f"[{i}/{total}]")

            preference = self.generator.generate(sample)

            preferences.append(preference)

        return preferences

In [66]:
import json

with open("evaluation_dataset.json", "r") as f:

    evaluation_dataset = json.load(f)

In [67]:
generator = PreferenceGenerator()

builder = PreferenceDatasetBuilder(generator)

preference_dataset = builder.build(
    evaluation_dataset
)

[1/1]


In [68]:
preference_dataset

[PreferenceExample(prompt='Clinical Question:\n\nA junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?\n\nOptions:\nA. Disclose the error to the patient and put it in the operative report\nB. Tell the attending that he cannot fail to disclose this mistake\nC. Report the physician to the ethics committee\nD. Refuse to dictate the operative report\n\nSelect the best answer.', chosen='B. Tell the attending that he cannot fail to disclose this m

In [69]:
from pprint import pprint

print(f"Total Preference Pairs: {len(preference_dataset)}")

pprint(
    preference_dataset[0].model_dump()
)

Total Preference Pairs: 1
{'chosen': 'B. Tell the attending that he cannot fail to disclose this mistake',
 'failure_capabilities': ['Clinical Reasoning', 'Ethics', 'Communication'],
 'primary_failure': 'Clinical Reasoning',
 'prompt': 'Clinical Question:\n'
           '\n'
           'A junior orthopaedic surgery resident is completing a carpal '
           'tunnel repair with the department chairman as the attending '
           'physician. During the case, the resident inadvertently cuts a '
           'flexor tendon. The tendon is repaired without complication. The '
           'attending tells the resident that the patient will do fine, and '
           'there is no need to report this minor complication that will not '
           'harm the patient, as he does not want to make the patient worry '
           'unnecessarily. He tells the resident to leave this complication '
           'out of the operative report. Which of the following is the correct '
           'next action for 

In [70]:
import json

with open(

    "preference_dataset.json",

    "w"

) as f:

    for example in preference_dataset:

        f.write(

            json.dumps(

                {

                    "prompt": example.prompt,

                    "chosen": example.chosen,

                    "rejected": example.rejected

                }

            )

            + "\n"

        )

print("Saved preference_dataset.json")

Saved preference_dataset.json
